In [1]:
import pandas as pd
import numpy as np
import time
import pickle

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report

from xgboost import XGBClassifier

In [2]:
import pandas as pd
import numpy as np
import re

# --- Load ISOT ---
fake_isot = pd.read_csv(r"C:\Users\wtpir\Documents\Datasets\Fake.csv")
true_isot = pd.read_csv(r"C:\Users\wtpir\Documents\Datasets\True.csv")

fake_isot['label'] = 0
true_isot['label'] = 1

# Strip Reuters tag
true_isot['text'] = true_isot['text'].str.replace(
    r'^[A-Z\s,\.]+\(Reuters\)\s*-\s*', '', regex=True
)

isot_df = pd.concat([fake_isot, true_isot], ignore_index=True)
isot_df = isot_df[['text', 'label']]

# --- Load WELFake ---
welfake_df = pd.read_csv(r"C:\Users\wtpir\Documents\Datasets\WELFake_Dataset.csv")
welfake_df = welfake_df[['text', 'label']].dropna()

# --- Combine both ---
df = pd.concat([isot_df, welfake_df], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df['content'] = df['text'].fillna('')
df = df.dropna(subset=['content', 'label'])

print("Combined dataset size:", len(df))
print(df['label'].value_counts())

Combined dataset size: 116993
label
0    58509
1    58484
Name: count, dtype: int64


In [3]:
X = df['content']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", len(X_train), "Test:", len(X_test))

Train: 93594 Test: 23399


In [4]:
vectorizer = TfidfVectorizer(
    max_features=100000,      # Increased to capture more vocabulary
    ngram_range=(1,3),        # Use unigrams, bigrams, and trigrams for more context
    stop_words='english',
    max_df=0.9,               # Ignore terms that appear in more than 90% of documents
    min_df=3                  # Include terms that appear in at least 1 document
)

print("Vectorizing...")
start = time.time()

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("Done in:", round(time.time() - start, 2), "sec")
print("Shape:", X_train_tfidf.shape)

Vectorizing...
Done in: 191.98 sec
Shape: (93594, 100000)


In [5]:
print("Training ONE strong model...\n")

model = XGBClassifier(
    n_estimators=800,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='hist',
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

start = time.time()
model.fit(X_train_tfidf, y_train)
train_time = round(time.time() - start, 2)

print(f"Training Time: {train_time}s")

best_model = model

Training ONE strong model...

Training Time: 1646.55s


In [6]:
y_pred = best_model.predict(X_test_tfidf)

print("FINAL Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

FINAL Accuracy: 0.6692593700585495

Classification Report:

              precision    recall  f1-score   support

           0       0.69      0.62      0.65     11702
           1       0.65      0.72      0.69     11697

    accuracy                           0.67     23399
   macro avg       0.67      0.67      0.67     23399
weighted avg       0.67      0.67      0.67     23399



In [7]:
pickle.dump(best_model, open("best_fake_news_model.pkl", "wb"))
pickle.dump(vectorizer, open("tfidf_vectorizer.pkl", "wb"))

print("Best model and vectorizer saved!")

Best model and vectorizer saved!


In [8]:
def predict_news(text, confidence_threshold=0.6):
    # Ensure the vectorizer and model are loaded if not already
    try:
        global vectorizer, best_model
        if 'vectorizer' not in globals() or 'best_model' not in globals():
            vectorizer = pickle.load(open("tfidf_vectorizer.pkl", "rb"))
            best_model = pickle.load(open("best_fake_news_model.pkl", "rb"))
    except FileNotFoundError:
        return "Error: Model or vectorizer not found. Please run training cells first."

    text_tfidf = vectorizer.transform([text])
    
    # Check if the text contains any words the model knows
    if text_tfidf.nnz == 0:
        return "Uncertain (No known words) ❓"
    
    # Get probability instead of just the class
    prob = best_model.predict_proba(text_tfidf)[0]
    pred = best_model.predict(text_tfidf)[0]
    
    # If the model is not confident, flag it
    # We take the probability of the predicted class
    predicted_class_prob = prob[pred] 
    if predicted_class_prob < confidence_threshold:
        return f"Uncertain (Confidence: {predicted_class_prob:.2f}) ❓"
        
    return "Real 🟢" if pred == 1 else "Fake 🔴"

In [10]:
tests = [

    # ================= FAKE 🔴 =================

    "BREAKING: Hillary Clinton ARRESTED by FBI agents!! Sources confirm Obama involved in MASSIVE COVER-UP. Share before they DELETE THIS!!",

    "SHOCK REPORT: Scientists ADMIT vaccines contain microchips!! Government hiding the TRUTH. WAKE UP PEOPLE!!",

    "Donald Trump is a TRAITOR who sold America to Russia!! Leaked tapes PROVE collusion with Putin. Deep state COVERING IT UP!!",

    "BOMBSHELL: George Soros paid protesters $1500 each to riot!! Mainstream media REFUSES to report this. SHARE BEFORE DELETED!!",

    "Obama ILLEGALLY spied on Trump campaign using CIA assets!! Biggest political scandal in US history being BURIED by fake news!!",


    # ================= REAL 🟢 =================

    "President Trump signed an executive order on Wednesday directing federal agencies to reduce regulations on small businesses, calling the move a boost to economic growth.",

    "The Senate voted 51 to 49 on Thursday to confirm the president's nominee to the federal appeals court, with all Democrats voting against the confirmation.",

    "Special counsel Robert Mueller has requested documents from the White House related to the firing of former national security adviser Michael Flynn.",

    "Congress approved a continuing resolution to fund the government through February, avoiding a shutdown after weeks of negotiations between Republican and Democratic leaders.",

    "The Trump administration announced new tariffs on steel and aluminum imports, a move that drew criticism from trading partners including Canada, Mexico and the European Union.",

]

print("\n--- Testing ---\n")
for t in tests:
    print(f"Text: {t}\nPrediction: {predict_news(t)}\n")


--- Testing ---

Text: BREAKING: Hillary Clinton ARRESTED by FBI agents!! Sources confirm Obama involved in MASSIVE COVER-UP. Share before they DELETE THIS!!
Prediction: Real 🟢

Text: SHOCK REPORT: Scientists ADMIT vaccines contain microchips!! Government hiding the TRUTH. WAKE UP PEOPLE!!
Prediction: Real 🟢

Text: Donald Trump is a TRAITOR who sold America to Russia!! Leaked tapes PROVE collusion with Putin. Deep state COVERING IT UP!!
Prediction: Real 🟢

Text: BOMBSHELL: George Soros paid protesters $1500 each to riot!! Mainstream media REFUSES to report this. SHARE BEFORE DELETED!!
Prediction: Real 🟢

Text: Obama ILLEGALLY spied on Trump campaign using CIA assets!! Biggest political scandal in US history being BURIED by fake news!!
Prediction: Real 🟢

Text: President Trump signed an executive order on Wednesday directing federal agencies to reduce regulations on small businesses, calling the move a boost to economic growth.
Prediction: Real 🟢

Text: The Senate voted 51 to 49 on Thu